In [1]:
from datasets import load_dataset

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate

from pydantic import BaseModel, Field

import numpy as np

from typing import Any
import json

d:\Dev\tcc\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:
# load LLM models
llm = ChatOpenAI(
    base_url='http://127.0.0.1:1234/v1',
    #model='google/gemma-4-e2b',
    model='deepseek/deepseek-r1-0528-qwen3-8b',
    api_key='none',
    temperature=0,
    extra_body={
        'reasoning' : {
            'effort' : 'low'
        }
    }
)

embed = OpenAIEmbeddings(
    base_url='http://127.0.0.1:1234/v1',
    model='text-embedding-bge-m3',
    api_key='none',
    check_embedding_ctx_length=False
)

In [3]:
# load qasper dataset
train_dataset = load_dataset(
    'allenai/qasper',
    split='train'
)

In [4]:
def parse_qasper_documents_ctx(example: dict) -> dict | None:
    title = example.get('title', None)
    if title is None:
        return None

    text = example.get('full_text', None)
    if text is None:
        return None

    sections = text.get('section_name', None)
    if sections is None:
        return None

    paragraphs = text.get('paragraphs', None)
    if paragraphs is None:
        return None

    return {
        'title' : title,
        'content' : [{'section' : sections[i], 'paragraphs' : [p for p in paragraphs[i] if p.strip()]} for i in range(len(sections))],
        'all_paragraphs' : [p for s in paragraphs for p in s if p.strip()]
    }

In [5]:
ex1 = parse_qasper_documents_ctx(train_dataset[0])

In [12]:
# helper function to parse hotpot qa documents
def parse_hotpotqa_ctx_documents(example: dict) -> list[dict]:
    ctx = example.get('context', None)
    if ctx is None:
        return []
    
    document_count = len(ctx.get('title', 0))
    if document_count == 0:
        return []
    
    out_documents = []
    for doc_i in range(document_count):
        out_documents.append({
            'title' : ctx['title'][doc_i],
            'sentences' : ctx['sentences'][doc_i]
        })

    return out_documents

In [14]:
# Custom RAKG implementation

# Entities
class Entity(BaseModel):
    name: str = Field(description='The main subject of the named entity')
    type: str = Field(description='The category of the subject')
    description: str = Field(description='A summary description explaining what the entity is')

class EntitiesResult(BaseModel):
    entities: list[Entity] = Field(description=('All named entities extracted from the text.'))

class EntityComparisonResult(BaseModel):
    result: bool

# KG
class KGAttribute(BaseModel):
    key: str = Field(description='The name of an attribute of the central entity.'),
    value: str = Field(description='The value of the attribute.')

class KGRelationship(BaseModel):
    relation: str = Field(description=('Relationship originating from the central entity and pointing to the target entity.'))
    target_name: str = Field(description='Canonical name of the target entity.')
    target_type: str = Field(description='Entity category of the target entity.')
    target_description: str = Field(description='Description of the target entity based on the provided context.')
    relation_description: str = Field(description=('Description explaining the relationship between the central entity and the target entity.'))

class KGCentralEntity(BaseModel):
    name: str = Field(description='Canonical name of the central entity.')
    type: str = Field(description='Entity category of the central entity.')
    description: str = Field(description=('Description of the central entity based on the provided context.'))
    attributes: list[KGAttribute] = Field(
        default_factory=list,
        description='Unique attributes of the central entity.'
    )
    relationships: list[KGRelationship] = Field(
        default_factory=list,
        description=('Unique outgoing relationships originating from the central entity.')
    )

class KnowledgeGraphResult(BaseModel):
    central_entity: KGCentralEntity

# Helpers
def get_named_entity_prompt() -> str:
    return '''
You are a named entity recognition assistant.

Text:
{text}

Task:
Extract all named entities mentioned in the text.

For each entity, provide:
- name: the canonical name of the entity.
- type: the most appropriate entity category.
- description: a concise description of the entity's role or significance in the provided text.

Guidelines:
1. Analyze the entire text before extracting entities.
2. Extract every relevant named entity explicitly mentioned in the text.
3. Include people, organizations, locations, products, software, datasets, methods, events, dates, and other named entities when appropriate.
4. The description should be specific to the context of the provided text, not a generic encyclopedia definition.
5. If an entity appears multiple times in the text, produce only one entry that summarizes all available information about that entity.
6. Do not extract common nouns, unnamed concepts, or generic references.
7. Do not infer information that is not supported by the text.
8. If the text contains no meaningful information or no named entities, return an empty entities list.
9. Return only the structured output defined by the schema.
'''.strip()

def get_entity_similarity_prompt() -> str:
    return '''
You are an entity resolution assistant.

Determine whether the following two JSON objects refer to the same real-world entity.

Entity 1:
{entity_1}

Entity 2:
{entity_2}

Rules:
1. Compare the canonical entity names.
2. Compare the entity types.
3. Use the descriptions only to resolve ambiguity.
4. Ignore differences in wording, formatting, capitalization, field order, and missing information.
5. If both entities have the same canonical name and compatible types, return True unless there is explicit evidence that they refer to different real-world entities.
6. Different descriptions do not imply different entities. They may describe different facts about the same entity.
7. Return False only when there is clear evidence that they represent different real-world entities.
'''.strip()

def get_knowledge_graph_prompt() -> str:
    return '''
You are a knowledge graph extraction assistant.

Text:
{text}

Target Entity:
{target_entity}

Related Knowledge Graphs:
{related_kg}

Task:
Extract a knowledge graph centered only on the target entity.

Rules:

1. Use only information explicitly supported by the provided text or the related knowledge graphs.

2. The output must describe only the target entity.

3. Every relationship must originate from the target entity.

4. Distinguish attributes from relationships:
   - Attributes are intrinsic properties of the target entity represented by literal values (text, numbers, dates, booleans, measurements, or other non-entity concepts).
   - Relationships connect the target entity to another identifiable named entity.
   - If the value of a fact is another named entity, it must always be represented as a relationship.
   - Attributes must never reference another named entity.
   - Never represent the same fact as both an attribute and a relationship.

5. Create an entity only if it:
   - is explicitly mentioned in the provided context;
   - has its own identity;
   - exists independently of the target entity;
   - can reasonably be represented as a standalone node in a knowledge graph.

6. Never create entities for actions, tasks, functionalities, capabilities, features, qualities, processes, descriptive phrases, or generic concepts, even if they appear as the object of a sentence. Represent such information as attributes or include it in descriptions.

7. Relationships must represent factual connections explicitly supported by the provided context. Do not infer new relationships.

8. Use canonical entity names and a single consistent type for each entity.

9. Entity descriptions must describe what the entity is, not the actions it performs or the relationships it participates in.

10. Use short, canonical relation names.

11. If the related knowledge graphs contain incoming relationships, generate the appropriate semantic inverse when necessary instead of copying the original predicate.

12. Remove duplicate attributes and relationships by merging semantically equivalent facts.
'''.strip()

# Classes

class Agents:
    def __init__(self, llm: ChatOpenAI, embed: OpenAIEmbeddings):
        self.llm = llm
        self.embed = embed
    
    def get_named_entities(self, text: str) -> list:
        # prepare prompt
        prompt = ChatPromptTemplate.from_template(get_named_entity_prompt())
        structured_llm = self.llm.with_structured_output(EntitiesResult)
        chain = prompt | structured_llm

        # call llm
        result = chain.invoke({'text' : text})
        return result.model_dump().get('entities', {})

    def get_context_vectors(self, context: list[str]):
        vectors = np.asarray(
            self.embed.embed_documents(context),
            dtype=np.float32
        )
        norms = np.linalg.norm(vectors, axis=1, keepdims=True)
        normalized_vectors = vectors / np.clip(norms, 1e-12, None)
        return normalized_vectors

    def get_top_k_context(self, query: str, vectors: np.ndarray, top_k: int = 5):
        query_vector = self.get_context_vectors([query])[0]
        similarities = query_vector @ vectors.T

        top_indices = np.argsort(similarities)[::-1][:top_k]
        return [
            (idx, similarities[idx])
            for idx in top_indices
        ]

    def __get_similar_entities_keys(self, entities: dict, threshold: float = 0.60) -> list[tuple[str, str]]:
        if len(entities) < 2:
            return []
        
        keys = list(entities.keys())
        texts = [f'{entity["name"]} {entity["type"]}' for entity in entities.values()]

        # similarity
        vectors = self.get_context_vectors(texts)
        similarity_matrix = vectors @ vectors.T

        row_indices, column_indices = np.where(np.triu(similarity_matrix, k=1) > threshold)
        return [
            (
                keys[i],
                keys[j],
                float(similarity_matrix[i, j]),
            )
            for i, j in zip(row_indices, column_indices)
        ]

    def get_similar_entities_keys(self, entities: dict) -> list[tuple[str, str]]:
        # Initial candidates, with pre filter
        candidates = self.__get_similar_entities_keys(entities)

        # prepare prompt
        prompt = ChatPromptTemplate.from_template(get_entity_similarity_prompt())
        structured_llm = self.llm.with_structured_output(EntityComparisonResult)
        chain = prompt | structured_llm

        out_candidates = []
        for ents in candidates:
            ent1 = entities.get(ents[0])
            ent2 = entities.get(ents[1])

            # validate entity similarity with llm
            result = chain.invoke({
                'entity_1': json.dumps(ent1, ensure_ascii=False, sort_keys=True),
                'entity_2': json.dumps(ent2, ensure_ascii=False, sort_keys=True),
            }).result

            if result:
                out_candidates.append(ents)

        return out_candidates

    def extract_knowledge_graph(self, target_entity: str, ctx_text: str, related_kg: list[dict[str, Any]] | dict[str, Any] | None = None) -> dict[str, Any]:
        if not ctx_text or not ctx_text.strip():
            raise ValueError('Context text must not be empty')

        if not target_entity or not target_entity.strip():
            raise ValueError('Target entoty must not be empty')

        related_kg = related_kg or []
        related_kg_text = json.dumps(
            related_kg,
            ensure_ascii=False,
            indent=2,
            default=str
        )

        prompt = ChatPromptTemplate.from_template(get_knowledge_graph_prompt())
        structured_llm = self.llm.with_structured_output(KnowledgeGraphResult)
        chain = prompt | structured_llm

        result = chain.invoke({
            'text' : ctx_text,
            'target_entity' : target_entity,
            'related_kg' : related_kg_text
        })

        return result.model_dump()

class CustomRAKG:
    def __init__(self, llm: ChatOpenAI, embed: OpenAIEmbeddings):
        self.agents = Agents(llm, embed)

    def __identify_entities(self, entities: list, entity_num: int) -> dict:
        identified_entities = {}
        for offset, entity in enumerate(entities):
            identified_entities.update({
                f'entity{entity_num + offset}' : entity
            })
        
        return identified_entities

    def get_preliminary_entities(self, sents: list[str]) -> dict:
        all_entities = {}
        entity_num = 1

        for i, sent in enumerate(sents):
            # extract entities
            entities = self.agents.get_named_entities(sent)
            if not entities:
                continue

            # add sentence referene to entity
            for j in range(len(entities)):
                entities[j]['sent_id'] = i

            # create unique key for entities
            identified_entities = self.__identify_entities(entities, entity_num)
            entity_num += len(entities)
            
            all_entities.update(identified_entities)

        return all_entities
    
    def __merge_entities(self, entities: dict):
        similar = self.agents.get_similar_entities_keys(entities)

        parent = {}

        def find(x):
            if parent[x] != x:
                parent[x] = find(parent[x])
            return parent[x]

        def union(x, y):
            root_x = find(x)
            root_y = find(y)
            if root_x != root_y:
                parent[root_y] = root_x

        # Initialize disjoint sets
        for ent in entities:
            parent[ent] = ent

        # Merge similar entities
        for a, b, _ in similar:
            if a in entities and b in entities:
                union(a, b)

        # Build groups
        groups = {}
        for ent in entities:
            root = find(ent)
            groups.setdefault(root, []).append(ent)

        merged_entities = {}

        # Merge each group
        for group in groups.values():
            main_ent = group[0]

            descriptions = []
            sent_ids = set()

            merged = entities[main_ent].copy()

            for ent in group:
                entity = entities[ent]

                # Merge descriptions
                desc = entity['description']
                if desc not in descriptions:
                    descriptions.append(desc)

                # Merge sentence ids
                sid = entity.get('sent_id')
                if isinstance(sid, list):
                    sent_ids.update(sid)
                elif sid is not None:
                    sent_ids.add(sid)

            merged['description'] = descriptions
            merged['sent_id'] = sorted(sent_ids)

            merged_entities[main_ent] = merged

        return merged_entities

    def get_kg_from_sents(self, sents: list[str]):
        # entity recognition
        preliminary_entities = self.get_preliminary_entities(sents)
        entities = self.__merge_entities(preliminary_entities)

        #return entities

        # build kg
        ctx_vectors = self.agents.get_context_vectors(sents)
        results = {}
        for id, ent in entities.items():
            sent_ids = set(ent.get('sent_id', []))
            ent_sents = [s for i, s in enumerate(sents) if i in sent_ids]
            if not ent_sents:
                print(f'Can\'t find sentences from entity: {ent}')
                continue

            name = ent.get('name', '')
            if not name:
                print(f'Can\'t find entity name from entity: {ent}')
                continue

            # retieve the top 5 context sentences
            top_ctx = self.agents.get_top_k_context(name, ctx_vectors, top_k=5)
            top_ctx_idx = set([int(i) for i, _ in top_ctx])

            # top context + entity sentences
            unique_ctx_idx = list(sent_ids | top_ctx_idx)
            ctx_sents = [s for i, s in enumerate(sents) if i in unique_ctx_idx]
            ctx_text = ', '.join(ctx_sents)

            result = self.agents.extract_knowledge_graph(name, ctx_text, None)
            results[id] = result

        return results

    def convert_kg(self, kg: dict):
        output = {
            'entities': [],
            'relations': []
        }

        entity_registry = {}
        relation_registry = set()

        # Register central entities
        for item in kg.values():
            central = item['central_entity']
            name = central['name']

            entity = entity_registry.setdefault(name, {
                'name': name,
                'type': central['type'],
                'description': central.get('description', ''),
                'attributes': {}
            })

            # Merge attributes
            for attr in central.get('attributes', []):
                entity['attributes'][attr['key']] = attr['value']

        # Register target entities and relations
        for item in kg.values():
            central = item['central_entity']
            source = central['name']

            for rel in central.get('relationships', []):
                target = rel['target_name']

                target_entity = entity_registry.setdefault(target, {
                    'name': target,
                    'type': rel['target_type'],
                    'description': rel.get('target_description', ''),
                    'attributes': {}
                })

                # Keep a single entity type
                if not target_entity.get('type'):
                    target_entity['type'] = rel['target_type']

                relation = (source, rel['relation'], target)
                if relation in relation_registry:
                    continue

                relation_registry.add(relation)

                output['relations'].append([
                    source,
                    rel['relation'],
                    target,
                    rel.get('relation_description', '')
                ])

        output['entities'] = sorted(
            entity_registry.values(),
            key=lambda x: x['name']
        )

        output['relations'] = sorted(
            output['relations'],
            key=lambda x: (x[0], x[1], x[2])
        )

        return output

In [15]:
agent = Agents(llm, embed)
crakg = CustomRAKG(llm, embed)

In [16]:
e1 = 'Google released Gemini 2.5 to improve reasoning and multimodal capabilities across its AI products.'
e2 = 'Several months later, Google integrated Gemini 2.5 into Google Workspace, allowing users to generate documents, summarize emails, and analyze spreadsheets.'

In [17]:
ents = crakg.get_kg_from_sents([e1, e2])

d:\Dev\tcc\.venv\Lib\site-packages\pydantic\json_schema.py:2463: PydanticJsonSchemaWarning: Default value (FieldInfo(annotation=NoneType, required=True, description='The name of an attribute of the central entity.'),) is not JSON serializable; excluding default from JSON schema [non-serializable-default]
  warnings.warn(message, PydanticJsonSchemaWarning)


In [18]:
print(json.dumps(
    ents,
    ensure_ascii=False,
    indent=2,
    default=str
))

{
  "entity1": {
    "central_entity": {
      "name": "Google",
      "type": "Organization",
      "description": "A multinational technology company that develops software and services.",
      "attributes": [
        {
          "key": "action",
          "value": "released Gemini 2.5"
        }
      ],
      "relationships": [
        {
          "relation": "owns_product",
          "target_name": "Gemini 2.5",
          "target_type": "Product",
          "target_description": "An AI product released by Google to improve reasoning and multimodal capabilities.",
          "relation_description": "Google owns the Gemini 2.5 product."
        }
      ]
    }
  },
  "entity2": {
    "central_entity": {
      "name": "Gemini 2.5",
      "type": "AI Model",
      "description": "An AI model released by Google to improve reasoning and multimodal capabilities across its AI products.",
      "attributes": [
        {
          "key": "release_date",
          "value": "Several months la

In [19]:
final_kg = crakg.convert_kg(ents)

In [20]:
print(json.dumps(
    final_kg,
    ensure_ascii=False,
    indent=2,
    default=str
))

{
  "entities": [
    {
      "name": "Gemini 2.5",
      "type": "AI Model",
      "description": "An AI model released by Google to improve reasoning and multimodal capabilities across its AI products.",
      "attributes": {
        "release_date": "Several months later after the initial release of Gemini",
        "purpose": "To improve reasoning and multimodal capabilities across Google's AI products"
      }
    },
    {
      "name": "Google",
      "type": "Organization",
      "description": "A multinational technology company that develops software and services.",
      "attributes": {
        "action": "released Gemini 2.5"
      }
    },
    {
      "name": "Google Workspace",
      "type": "Software Suite",
      "description": "A suite of productivity tools developed by Google, including applications for document generation, email summarization, and spreadsheet analysis.",
      "attributes": {
        "Integration Date": "Several months after the release of Gemini 2.5",
